# Example: Stylized Facts of Log Growth Rate Data

Financial returns exhibit universal statistical patterns, called _stylized facts_, that deviate from the assumptions of classical models. Understanding these facts is essential for building realistic portfolio models and stress tests. In this example, we compute log growth rates from synthetic training data and examine three key properties: fat-tailed distributions, near-zero autocorrelation of returns, and persistent volatility clustering.

> __Learning Objectives:__
>
> * __Log Growth Rate Computation:__ Compute the continuously compounded growth rate (CCGR) matrix from synthetic price data. Explore the statistical properties of the growth rate time series and its distribution.
> * __Fat Tails and Non-Normality:__ Test whether growth rates follow a Normal, Laplace, or Student-t distribution using the Anderson-Darling test. Estimate the tail index via Hill's estimator to quantify how heavy the tails are.
> * __Autocorrelation and Volatility Clustering:__ Verify that raw returns are approximately uncorrelated, consistent with the random walk hypothesis. Show that absolute returns exhibit persistent positive autocorrelation, the signature of volatility clustering.

Let's dive in and compute these stylized facts from our synthetic data!
___

## Setup, Data and Prerequisites
We begin by loading the `eCornellAIFinance` package and the frozen synthetic training dataset via `Include.jl`.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

### Constants


In [ ]:
# Time-series inspection configuration
Δt = 1.0 / 252.0
ticker_to_visualize = "MARKET"
STUDENT_T_NU_GRID = [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0, 15.0, 20.0, 30.0]
MAX_ACF_LAG = 125


### Data
We load the frozen 20-year synthetic training dataset using the [`MySyntheticTrainingDataSet()` function](../../code/src/Files.jl) from the `eCornellAIFinance` package. This dataset contains daily close prices for 424 tickers plus a synthetic market index, generated from the pre-trained JumpHMM portfolio surrogate, with a curated market path exhibiting realistic drawdowns, jump clusters, and an 8.3% CAGR.

In [ ]:
# Load the frozen synthetic training dataset from the eCornellAIFinance package.
# Returns a Dict with keys: "dataset" (ticker → DataFrame), "tickers" (sorted list),
# "n_days" (total trading days), "n_years" (span in years).
training_data = MySyntheticTrainingDataSet();

# Extract the dataset dictionary: maps each ticker symbol to a DataFrame
# with columns including :close (daily closing prices)
dataset = training_data["dataset"];

# Extract the sorted list of 424 ticker symbols
list_of_tickers = training_data["tickers"];

# Verify the data dimensions
println("Loaded $(length(list_of_tickers)) tickers × $(training_data["n_days"]) days ($(training_data["n_years"]) years)")

___
## Task 1: Compute the Log Growth Rate Matrix
We compute the continuously compounded growth rate (CCGR) for every ticker in the dataset. The growth rate of asset $i$ between time $j-1$ and $j$ is:

$$\boxed{g^{(i)}_{j,j-1} = \left(\frac{1}{\Delta t}\right)\,\ln\left(\frac{S^{(i)}_j}{S^{(i)}_{j-1}}\right)}$$

where $\Delta t = 1/252$ for daily data (one trading day in years).

> __What are we going to do?__
>
> We compute the growth rate matrix $\mathbf{X} \in \mathbb{R}^{(T-1) \times N}$ where each row is a time step and each column is a ticker. Then we visualize the growth rate time series and its distribution for a selected ticker.

The growth rate time series also reveals the distribution shape and temporal dynamics of each asset.

> __What should we see?__
>
> The growth rate appears as a stationary random process with non-periodic bursts of extreme volatility followed by calm periods. The distribution has a sharp peak near zero and heavy tails compared to a Normal distribution. This is the "fat tails" stylized fact.

In the next cell, we compute the full growth rate matrix and store the result in the `growth_rate_array::Matrix{Float64}` variable.

In [ ]:
growth_rate_array = let

    # Setup: define the time step and array dimensions
    N = length(list_of_tickers); # number of tickers (columns)
    T_total = training_data["n_days"]; # total number of price observations
    T = T_total - 1;             # number of growth rate observations (one fewer than prices)

    # Allocate the growth rate matrix: rows = time steps, columns = tickers
    X = zeros(T, N);

    # Compute the CCGR for each ticker: g = (1/Δt) * ln(S_{t+1} / S_t)
    for (j, ticker) ∈ enumerate(list_of_tickers)

        # Extract the daily closing prices for this ticker
        prices = dataset[ticker][:, :close];

        # Compute the log growth rate for each consecutive pair of prices
        for t ∈ 1:T
            X[t, j] = (1.0 / Δt) * log(prices[t + 1] / prices[t]);
        end
    end

    # Sanity check: verify the output dimensions match expectations
    @assert size(X) == (T, N) "Growth rate matrix should be $(T) × $(N)"
    println("Growth rate matrix: $(T) days × $(N) tickers")
    println("Mean growth rate (annualized, all tickers): $(round(mean(X) * 100, digits=2))%")

    X  # return the (T-1) × N growth rate matrix
end

### Visualize
Select a ticker to explore. Try different tickers to see how the growth rate behavior varies across assets.

The selected ticker now lives in the `### Constants` subsection above. In the next cell we extract its column from `growth_rate_array` into the `X_ticker::Vector{Float64}` variable for downstream tasks.


In [ ]:
# Extract the column from growth_rate_array; the index lookup is scoped locally.
X_ticker = let
    idx = findfirst(==(ticker_to_visualize), list_of_tickers);
    @assert idx !== nothing "Ticker $(ticker_to_visualize) not found in list_of_tickers"
    X = growth_rate_array[:, idx];
    println("Selected $(ticker_to_visualize): $(length(X)) daily growth rates")
    X
end;

**Growth rate time series and distribution:** The left panel shows the daily growth rate over 20 years. The right panel shows the empirical density compared to fitted Normal, Laplace, and Student-t distributions.

> __What should we see?__
>
> The time series shows bursts of high volatility (large positive and negative spikes clustered together) interspersed with calmer periods, this is volatility clustering. The distribution is sharply peaked near zero with heavy tails. The Laplace distribution should provide the best fit (reflecting the [Laplace-distributed latent variables in the JumpHMM](https://arxiv.org/pdf/2603.10202)), followed by Student-t, with Normal being the worst.

In the next cell, we use [`fit_mle` from Distributions.jl](https://juliastats.org/Distributions.jl/stable/fit/) to estimate parameters for Normal, Laplace, and Student-t distributions, then plot the fitted densities against the empirical distribution.

In [ ]:
let
    # --- Step 1: Use the pre-computed growth rates for the selected ticker ---
    X = X_ticker;

    # --- Step 2: Plot the growth rate time series (left panel) ---
    p1 = plot(X, label="", lw=1, color=:red, alpha=0.8, linetype=:steppost,
        xlabel="Trading Day Index (AU)", ylabel="Growth rate g $(ticker_to_visualize) (1/yr)",
        title="$(ticker_to_visualize) Daily Growth Rate",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);

    # --- Step 3: Fit candidate distributions via MLE ---
    # Normal: fit mean and variance by maximum likelihood
    d_normal = fit_mle(Normal, X);

    # Laplace: fit location and scale by maximum likelihood
    d_laplace = fit_mle(Laplace, X);

    # Student-t: profile MLE, standardize data, then grid-search over degrees of freedom ν.
    # We search a discrete grid because the Student-t likelihood in ν is not convex.
    μ_X = mean(X); σ_X = std(X);
    Z = (X .- μ_X) ./ σ_X;  # standardized residuals
    best_ν = 5.0; best_ll = -Inf;
    for ν_cand ∈ STUDENT_T_NU_GRID
        ll = sum(logpdf.(TDist(ν_cand), Z));  # log-likelihood of standardized data
        if ll > best_ll
            best_ll = ll; best_ν = ν_cand;
        end
    end

    # Reconstruct the fitted Student-t with original location and scale.
    # Scale factor sqrt((ν-2)/ν) ensures the variance of LocationScale matches σ_X².
    d_tdist = LocationScale(μ_X, σ_X * sqrt((best_ν - 2)/best_ν), TDist(best_ν));

    # --- Step 4: Plot density comparison (right panel) ---
    p2 = plot(d_normal, lw=3, color=:deepskyblue1, label="Normal (MLE)",
        xlabel="Growth rate g $(ticker_to_visualize) (1/yr)", ylabel="Density (AU)",
        title="$(ticker_to_visualize) Distribution",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    plot!(p2, d_laplace, lw=3, color=:grey34, label="Laplace (MLE)");
    plot!(p2, d_tdist, lw=3, color=:green4, ls=:dash, label="Student-t (ν=$(Int(best_ν)))");
    density!(p2, X, lw=3, color=:red, label="Observed (daily)");  # empirical kernel density

    # --- Step 5: Combine into a side-by-side layout ---
    plot(p1, p2, layout=(1, 2), size=(1100, 420), margin=5Plots.mm)
end

___
## Task 2: Which Distribution Best Describes Growth Rates?
One of the most robust stylized facts of financial returns is that they are **not normally distributed**. Return distributions have fat tails. The density near zero is greater than Normal, and extreme events occur far more often than a Gaussian model predicts. But which distribution fits best?

> __What are we going to do?__
>
> We use the [`OneSampleADTest` from HypothesisTests.jl](https://juliastats.org/HypothesisTests.jl/stable/nonparametric/) to compare three candidate distributions (Normal, Laplace, and Student-t) by examining their $A^2$ test statistics. Lower $A^2$ means better fit. With large samples ($T \approx 5000$), the AD test has enormous power and may reject _all_ parametric models, so we focus on **relative fit** (which $A^2$ is smallest) rather than binary pass/fail.

The relative ranking reveals which parametric family best captures the empirical distribution shape.

> __What should we see?__
>
> The Normal should have the largest $A^2$ (worst fit). Laplace should have the smallest (best fit) because the [JumpHMM latent variables are Laplace-distributed](https://arxiv.org/pdf/2603.10202), shaping the observed growth rate distribution. Student-t falls between them despite driving the emission model. We also compute Hill's tail index $\alpha$ to quantify how heavy the tails are.

In the next cell, we fit the three distributions and run the Anderson-Darling test for each.

In [ ]:
let
    # --- Step 1: Use the pre-computed growth rates for the selected ticker ---
    μ = X_ticker;
    T = length(μ);

    # --- Step 2: Fit three candidate distributions via MLE ---
    d_normal = fit_mle(Normal, μ);    # Normal(μ̂, σ̂)
    d_laplace = fit_mle(Laplace, μ);  # Laplace(μ̂, b̂)

    # Student-t: profile MLE over degrees of freedom ν
    μ_X = mean(μ); σ_X = std(μ);
    Z = (μ .- μ_X) ./ σ_X;           # standardize for grid search
    best_ν = 5.0; best_ll = -Inf;
    for ν_cand ∈ STUDENT_T_NU_GRID
        ll = sum(logpdf.(TDist(ν_cand), Z));
        if ll > best_ll
            best_ll = ll; best_ν = ν_cand;
        end
    end
    d_tdist = LocationScale(μ_X, σ_X * sqrt((best_ν - 2)/best_ν), TDist(best_ν));

    # --- Step 3: Run the Anderson-Darling goodness-of-fit test for each distribution ---
    # The AD test computes the A² statistic: lower A² = better fit to the data.
    # With T ≈ 5000, the test has enormous power. All models may be formally rejected.
    # We focus on relative ranking (which A² is smallest) rather than binary pass/fail.
    ad_normal = OneSampleADTest(μ, d_normal);
    ad_laplace = OneSampleADTest(μ, d_laplace);
    ad_tdist = OneSampleADTest(μ, d_tdist);

    # --- Step 4: Build a comparison table sorted by A² (best fit first) ---
    results = [(name="Normal", A²=ad_normal.A², pval=pvalue(ad_normal)),
               (name="Laplace", A²=ad_laplace.A², pval=pvalue(ad_laplace)),
               (name="Student-t (ν=$(Int(best_ν)))", A²=ad_tdist.A², pval=pvalue(ad_tdist))];
    sort!(results, by=x -> x.A²);  # sort ascending: best fit on top

    # Format results into a DataFrame for display
    df = DataFrame(
        "Distribution" => [r.name for r ∈ results],
        "A² statistic" => [round(r.A², digits=2) for r ∈ results],
        "p-value" => [r.pval < 1e-6 ? "<1e-6" : string(round(r.pval, digits=4)) for r ∈ results],
        "Rank" => ["★ BEST", "2nd", "3rd"]
    );

    # --- Step 5: Display the ranked comparison table ---
    println("Anderson-Darling Goodness-of-Fit Comparison for $(ticker_to_visualize) (T = $(T)):")
    pretty_table(df, backend=:text, fit_table_in_display_horizontally=false,
        fit_table_in_display_vertically=false, 
        table_format=TextTableFormat(borders=text_table_borders__compact))
    println("\nNote: With T = $(T) observations, the AD test has very high power.")
    println("Focus on the A² ranking (lower = better fit), not just pass/fail.")
end

**Hill's Tail Index:** The tail index $\alpha$ quantifies how heavy the tails of the distribution are. For a Normal distribution, $\alpha \to \infty$. For financial returns, $\alpha$ is typically between 2 and 5, meaning extreme events occur _much_ more often than a Gaussian model predicts.

> __Hill's estimator__
>
> Uses only the top $k$ order statistics (not the entire sample). Given sorted observations $z_{(1)} \geq z_{(2)} \geq \ldots$, the estimator is:
>
> $$\boxed{\hat{\gamma}_k = \frac{1}{k}\sum_{i=1}^{k}\ln\frac{z_{(i)}}{z_{(k+1)}}, \qquad \hat{\alpha} = \frac{1}{\hat{\gamma}_k}}$$
>
> where $k \ll n$ controls how deep into the tail we look. A common choice is $k = \lfloor\sqrt{n}\rfloor$. Too large $k$ includes body observations; too small $k$ gives high variance.

In the next cell, we compute Hill's tail index on the positive tail of the growth rate data.

In [ ]:
let
    # --- Step 1: Use the pre-computed growth rates and isolate the positive tail ---
    μ = X_ticker;

    # Hill's estimator works on positive values only (right tail).
    # For symmetric distributions, the left tail has the same index.
    pos = filter(>(0), μ);
    n = length(pos);
    z = sort(pos, rev=true);  # descending order: z₍₁₎ ≥ z₍₂₎ ≥ ... ≥ z₍ₙ₎

    # --- Step 2: Compute the Hill estimator ---
    # Use the top k = ⌊√n⌋ order statistics (a standard bias-variance tradeoff).
    # Too large k includes body observations (bias up); too small k gives high variance.
    k = floor(Int, sqrt(n));

    # γ̂ₖ = (1/k) Σᵢ₌₁ᵏ ln(z₍ᵢ₎ / z₍ₖ₊₁₎), the Hill gamma estimate
    γ̂ = (1.0/k) * sum(log(z[i] / z[k+1]) for i ∈ 1:k);

    # α̂ = 1/γ̂, the tail index (power-law exponent)
    α_hill = 1.0 / γ̂;

    # --- Step 3: Display results with reference values ---
    println("Hill's tail index for $(ticker_to_visualize):")
    println("  k = $(k) (top order statistics out of $(n) positive observations)")
    println("  γ̂ = $(round(γ̂, digits=4))")
    println("  α̂ = $(round(α_hill, digits=2))")
    println()
    println("  Reference: Normal → ∞, Cauchy → 1, typical equity returns → 2-5")
end

___
## Task 3: Autocorrelation and Volatility Clustering
Two more stylized facts: (1) raw returns are approximately **uncorrelated** (consistent with the random walk hypothesis), and (2) absolute returns show persistent **positive autocorrelation**. This is volatility clustering, where periods of high volatility tend to follow periods of high volatility.

> __What are we going to do?__
>
> We compute the autocorrelation function (ACF) using the [`autocor` function from StatsBase.jl](https://juliastats.org/StatsBase.jl/stable/signalcorr/) for both raw growth rates $g_i(t)$ and absolute growth rates $|g_i(t)|$. We compare to the 99% confidence band under the null hypothesis of no autocorrelation.

The ACF comparison between raw and absolute returns is the key diagnostic for volatility clustering.

> __What should we see?__
>
> The raw ACF should hover near zero at all lags (random walk). The absolute ACF should be significantly positive for short lags (up to 10–50 days), decaying slowly. This is the volatility clustering signature that motivates models like JumpHMM.

**Autocorrelation of raw returns:** Under the random walk hypothesis, the ACF should be zero for all lags > 0. Near-zero autocorrelation at all lags confirms that daily returns are approximately unpredictable. We cannot use yesterday's return to forecast today's.

In the next cell, we compute and plot the ACF of raw growth rates with 99% confidence bands.

In [ ]:
let
    # --- Step 1: Use the pre-computed growth rates for the selected ticker ---
    X = X_ticker;
    T = length(X);

    # --- Step 2: Compute the autocorrelation function (ACF) for raw returns ---
    # ACF at lag h measures the linear correlation between g(t) and g(t+h).
    # Under the random walk hypothesis, ACF(h) ≈ 0 for all h > 0.
    max_lag = MAX_ACF_LAG;  # approximately 6 months of trading days
    lags = collect(0:max_lag);
    acf_raw = autocor(X, lags);  # sample autocorrelation at each lag

    # --- Step 3: Compute the 99% confidence band ---
    # Under H₀ (white noise), ACF(h) ~ N(0, 1/T) for large T.
    # The 99% band is ±2.576/√T (two-sided z-critical value for α = 0.01).
    conf_99 = 2.576 / sqrt(T);
    LINE = conf_99 * ones(length(lags));

    # --- Step 4: Plot the ACF with confidence bands ---
    plot(lags, acf_raw, lw=2, color=:red, label="Observed $(ticker_to_visualize)",
        linetype=:steppost,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent,
        xlabel="Lag (trading day)", ylabel="Autocorrelation Growth Rate",
        title="ACF of Raw Returns: $(ticker_to_visualize)", size=(750, 420));
    scatter!(lags, acf_raw, color=:red, msc=:red, ms=3, label="");
    plot!(lags, LINE, lw=2, color=:black, ls=:dash, label="99% confidence");
    plot!(lags, -LINE, lw=2, color=:black, ls=:dash, label="")
end

**Volatility clustering (ACF of |gₜ|):** Volatility clustering means that large moves (positive or negative) tend to cluster together. We measure this by computing the ACF of the _absolute_ growth rate $|g_i(t)|$.

> __What should we see?__
>
> Positive autocorrelation that decays slowly over tens or hundreds of lags, far exceeding the 99% confidence bands. This is one of the strongest and most universal stylized facts. It means: if today was volatile, tomorrow is more likely to be volatile too.

In the next cell, we compute and plot the ACF of absolute growth rates to visualize this clustering effect.

In [ ]:
let
    # --- Step 1: Use the pre-computed growth rates for the selected ticker ---
    X = X_ticker;
    T = length(X);

    # --- Step 2: Compute the ACF of absolute returns |g(t)| ---
    # Absolute returns measure volatility magnitude regardless of direction.
    # Persistent positive ACF at short lags indicates volatility clustering:
    # large moves (up or down) tend to be followed by large moves.
    max_lag = MAX_ACF_LAG;  # approximately 6 months of trading days
    lags = collect(0:max_lag);
    acf_abs = autocor(abs.(X), lags);  # ACF of |g(t)|

    # --- Step 3: Compute the 99% confidence band (same as raw ACF) ---
    conf_99 = 2.576 / sqrt(T);
    LINE = conf_99 * ones(length(lags));

    # --- Step 4: Plot, expect significant positive ACF decaying slowly ---
    plot(lags, acf_abs, lw=2, color=:red, label="Observed $(ticker_to_visualize)",
        linetype=:steppost,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent,
        xlabel="Lag (trading day)", ylabel="Autocorrelation |Growth Rate|",
        title="Volatility Clustering: ACF of |gₜ| for $(ticker_to_visualize)", size=(750, 420));
    plot!(lags, LINE, lw=2, color=:black, ls=:dash, label="99% confidence");
    plot!(lags, -LINE, lw=2, color=:black, ls=:dash, label="")
end

___
## Summary
Financial growth rates exhibit fat tails, near-zero autocorrelation, and persistent volatility clustering. These three universal stylized facts violate the assumptions of classical Normal return models.

> __Key Takeaways:__
>
> * **Growth rates are not normally distributed:** The Anderson-Darling test ranks Normal as the worst fit, with Laplace best capturing the fat-tailed, peaked shape of empirical returns. With large samples, focus on relative fit ranking rather than binary reject or fail-to-reject.
> * **Raw returns are approximately uncorrelated:** The autocorrelation function of raw growth rates drops to near zero at all lags, consistent with the random walk hypothesis. Past returns cannot be used to predict future returns.
> * **Volatility clusters persistently:** The autocorrelation of absolute growth rates shows significant positive values that decay slowly over tens of days. Large moves beget large moves, motivating regime-switching models like JumpHMM that capture this temporal dependence.

These properties motivate the use of regime-switching models like JumpHMM that capture non-Gaussian distributions and time-varying volatility for realistic portfolio analysis.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.
___